In [0]:
%run ./00_utils

In [0]:
import pyspark.sql.functions as F
from pyspark.sql.window import Window

In [0]:
df_programmi = spark.table('ta_coll.whatif.programmi')

In [0]:
df_programmi.select('Canale','ORA_INIZIO_TRX', 'Share').withColumn('ORA_INIZIO_TRX', F.round(F.col('ORA_INIZIO_TRX') / 60 / 60 / 10, 1)* 10).groupBy('ORA_INIZIO_TRX', "Canale").agg(F.median('Share')).display()

Databricks visualization. Run in Databricks to view.

In [0]:
df_programmi.select('Canale','Share').groupBy('Canale').agg(F.median('Share')).display()

Databricks visualization. Run in Databricks to view.

In [0]:
df_programmi.select('DES_GENERE_ESTESA_INT','Share').where(F.col('Canale').isin(['Italia 1','Canale 5', 'Rai 1', 'Rai 2', 'Rai 3', 'Tv8'])).groupBy('DES_GENERE_ESTESA_INT').agg(F.median('Share')).display()

Databricks visualization. Run in Databricks to view.

In [0]:
df_programmi.select('DES_GENERE_SPORT_INT','Share').where(F.col('DES_GENERE_SPORT_INT') != 'Nessun Genere').where(F.col('Canale').isin(['Italia 1','Canale 5', 'Rete 4', 'Rai 1', 'Rai 2', 'Rai 3', 'Tv8'])).groupBy('DES_GENERE_SPORT_INT').agg(F.median('Share')).display()

Databricks visualization. Run in Databricks to view.

In [0]:
df_programmi.select('DES_MANIFESTAZIONE_SPORT_INT','Share').where(F.col('DES_MANIFESTAZIONE_SPORT_INT') != 'Nessun Genere').where(F.col('Canale').isin(['Italia 1','Canale 5', 'Rete 4', 'Rai 1', 'Rai 2', 'Rai 3', 'Tv8'])).groupBy('DES_MANIFESTAZIONE_SPORT_INT').agg(F.median('Share')).display()

Databricks visualization. Run in Databricks to view.

In [0]:
df_programmi.select('DES_GENERE_FILM_INT','Share').where(F.col('DES_GENERE_FILM_INT') != 'GENERE CINEMAT. NON ATTRIBUITO').where(F.col('Canale').isin(['Italia 1','Canale 5', 'Rete 4' 'Rai 1', 'Rai 2', 'Rai 3', 'Tv8'])).groupBy('DES_GENERE_FILM_INT').agg(F.median('Share')).display()

Databricks visualization. Run in Databricks to view.

In [0]:
df_programmi.select('Canale','ETA_MEDIA').groupBy('Canale').agg(F.mean('ETA_MEDIA')).display()

Databricks visualization. Run in Databricks to view.

In [0]:
df_programmi.withColumn('duration', F.col("ORA_FINE_TRX") - F.col('ORA_INIZIO_TRX')).where(F.col('Canale').isin(['Italia 1','Canale 5', 'Rete 4', 'Rai 1', 'Rai 2', 'Rai 3', 'Tv8'])).select('duration', 'Share').display()

Databricks visualization. Run in Databricks to view.

In [0]:
from pyspark.sql.window import Window

w_rank = Window.partitionBy('Canale', 'Programma', 'Ora', 'GiornoSettimana').orderBy(F.col("data_days").desc())
w_time = Window.partitionBy('Canale', 'Programma', 'Ora', 'GiornoSettimana').orderBy("data_days").rangeBetween(Window.unboundedPreceding, -7)

storico_share = (
    df_programmi.where(F.col('Canale').isin(['Italia 1','Canale 5', 'Rete 4', 'Rai 1', 'Rai 2', 'Rai 3', 'Tv8', 'La7',' Nove'])).where(F.col('ORA_INIZIO_TRX')>25000).withColumn('Ora', F.round(F.col('ORA_INIZIO_TRX') / 60 / 60 / 10, 1) * 10)
    .withColumn('GiornoSettimana', F.dayofweek(F.col("Data")))
    .withColumn(
        "data_days",
        F.unix_date(F.col("Data"))
    ).withColumn("rank_back", F.row_number().over(w_rank)).where(F.col("rank_back") <= 15).withColumn(
        "median_share_last_5_7d",
        F.percentile_approx("Share", 0.5).over(w_time)
    ).withColumn('medianLiveVOSDAL', F.percentile_approx("LiveVOSDAL", 0.5).over(w_time)).select('Share','median_share_last_5_7d', 'medianLiveVOSDAL')
)

storico_share.display()

Databricks visualization. Run in Databricks to view.